# OSeMOSYS for Sierra Leone - Long Term Planning with Seasonality

## Setup and Pre-Processing

In [1]:
import os
from pathlib import Path

# === USER INPUT ===
model_name = "Karpowership Production Scenario with Seasonality"  # Change to any model folder inside 'models/'
# ===================

# === Set Dynamic File Paths ===
base_dir = Path("models") / model_name
data_file = base_dir / "data.txt"
model_file = base_dir / "model.txt"
solution_file = base_dir / "solution.txt"

# Output folders
res_dir = Path("res/csv")

# Ensure output directories exist
solution_file.parent.mkdir(parents=True, exist_ok=True)
res_dir.mkdir(parents=True, exist_ok=True)

## Running the Model

In [2]:
import subprocess

# Command to run GLPK
cmd = ["glpsol", "-m", model_file, "-d", data_file, "-o", solution_file]

# Run and capture output
try:
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    print("Solver ran successfully!")
    print(result.stdout)  # See solver output
except subprocess.CalledProcessError as e:
    print("Solver failed!")
    print(e.stdout)       # Show solver standard output
    print(e.stderr)       # Show solver error log


Solver ran successfully!
GLPSOL: GLPK LP/MIP Solver, v4.65
Parameter(s) specified in the command line:
 -m models\Karpowership Production Scenario with Seasonality\model.txt -d
 models\Karpowership Production Scenario with Seasonality\data.txt -o models\Karpowership Production Scenario with Seasonality\solution.txt
Reading model section from models\Karpowership Production Scenario with Seasonality\model.txt...
185 lines were read
Reading data section from models\Karpowership Production Scenario with Seasonality\data.txt...
300 lines were read
Generating OFL_Cost...
Generating Acc1_FuelProductionByTechnology...
Generating Acc2_FuelUseByTechnology...
Generating Acc3_AverageAnnualRateOfActivity...
Generating CAa1_TotalNewCapacity...
Generating CAa2_TotalAnnualCapacity...
Generating CAa5_TotalNewCapacity...
Generating CC1_UndiscountedCapitalInvestment...
Generating E2_AnnualEmissionProduction...
Generating EBa10_EnergyBalanceEachTS4...
Generating EBa1_RateOfFuelProduction1...
Generating EB

## Post Processing

In [3]:
# === Helper Function ===
def run_command(cmd, success_msg, fail_msg, exit_on_fail=False, ignore_errors=False):
    print(f"\n🔄 Running: {' '.join(map(str, cmd))}")
    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"✅ {success_msg}")
        return result.stdout
    except subprocess.CalledProcessError as e:
        if ignore_errors:
            print(f"⚠️ Partial success: {fail_msg} — continuing anyway.")
            print("STDOUT:\n", e.stdout)
            return e.stdout
        else:
            print(f"❌ {fail_msg}")
            print("STDOUT:\n", e.stdout)
            print("STDERR:\n", e.stderr)
            if exit_on_fail:
                exit(1)
            return None

# === Step 1: Setup config.yaml ===
if not os.path.exists("config.yaml"):
    run_command(
        ["otoole", "setup", "config", "config.yaml"],
        "✅ config.yaml created successfully!",
        "❌ Failed to generate config.yaml",
        exit_on_fail=True
    )
else:
    run_command(
        ["otoole", "setup", "config", "config.yaml", "--overwrite"],
        "✅ config.yaml overwritten successfully!",
        "❌ Failed to overwrite config.yaml"
    )

# === Step 2: Validate model ===
run_command(
    ["otoole", "validate", "datafile", str(data_file), "config.yaml"],
    "✅ Model data validated.",
    "⚠️ Validation warnings or errors occurred."
)

# === Step 3A: Convert data.txt (inputs) to CSV ===
run_command(
    [
        "otoole", "convert", "datafile", "csv",
        str(data_file), str(res_dir), "config.yaml"
    ],
    f"✅ Inputs extracted to: {res_dir}",
    "❌ Failed to convert inputs"
)

# === Step 3B: Extract solution.txt (outputs) to CSV ===
run_command(
    [
        "otoole", "results", "glpk", "csv",
        str(solution_file),
        str(res_dir),
        "datafile",
        str(data_file),
        "config.yaml",
        "--glpk_model", str(model_file)
    ],
    f"✅ Results extracted to: {res_dir}",
    "⚠️ Some results could not be extracted",
    ignore_errors=True  # Let it continue even if some result vars fail
)

print(f"\n🎉 Full pipeline completed for model: {model_name}")


🔄 Running: otoole setup config config.yaml --overwrite
✅ ✅ config.yaml overwritten successfully!

🔄 Running: otoole validate datafile models\Karpowership Production Scenario with Seasonality\data.txt config.yaml
✅ ✅ Model data validated.

🔄 Running: otoole convert datafile csv models\Karpowership Production Scenario with Seasonality\data.txt res\csv config.yaml
✅ ✅ Inputs extracted to: res\csv

🔄 Running: otoole results glpk csv models\Karpowership Production Scenario with Seasonality\solution.txt res\csv datafile models\Karpowership Production Scenario with Seasonality\data.txt config.yaml --glpk_model models\Karpowership Production Scenario with Seasonality\model.txt
⚠️ Partial success: ⚠️ Some results could not be extracted — continuing anyway.
STDOUT:
 ParserError: Error tokenizing data. C error: Expected 5 fields in line 18, saw 10



🎉 Full pipeline completed for model: Karpowership Production Scenario with Seasonality


## Results Visualization

### Colors

In [5]:
import pandas as pd
import json

# === Step 1: Load technologies from CSV ===
tech_file = res_dir / "TECHNOLOGY.csv"  
df_tech = pd.read_csv(tech_file)
technologies = sorted(df_tech["VALUE"].dropna().unique())  

# === Step 2: Assign colors using standard energy palette ===
tech_colors = {}

for tech in technologies:
    tech_lower = tech.lower()

    if "solar" in tech_lower:
        tech_colors[tech] = "#FFD700"  # Yellow
    elif "large_hydro" in tech_lower:
        tech_colors[tech] = "#1f77b4"  # Dark Blue
    elif "medium_hydro" in tech_lower:
        tech_colors[tech] = "#4da6ff"  # Medium Blue
    elif "small_hydro" in tech_lower:
        tech_colors[tech] = "#87CEFA"  # Light Blue
    elif "karpowership" in tech_lower or "hfo" in tech_lower:
        tech_colors[tech] = "#000000"  # Black
    elif "lfo" in tech_lower:
        tech_colors[tech] = "#8B4513"  # Brown
    elif "_prod" in tech_lower:
        tech_colors[tech] = "#A9A9A9"  # Gray (fuel producers)
    else:
        tech_colors[tech] = "#D3D3D3"  # Light gray fallback

# === Step 3: Print & export ===
print("\n🎨 Assigned technology colors:")
for tech, color in tech_colors.items():
    print(f"{tech}: {color}")

# Save to JSON
with open("tech_colors.json", "w") as f:
    json.dump(tech_colors, f, indent=2)


🎨 Assigned technology colors:
HFO_Prod: #000000
Karpowership: #000000
LFO_Plant: #8B4513
LFO_Prod: #8B4513
Large_Hydro: #1f77b4
Medium_Hydro: #4da6ff
Small_Hydro: #87CEFA
Solar_PV: #FFD700


### Total Annual Installed Capacity

In [6]:
import pandas as pd
import plotly.graph_objects as go

# === Load total and residual capacity data ===
df_total = pd.read_csv(res_dir / "TotalCapacityAnnual.csv")
df_resid = pd.read_csv(res_dir / "ResidualCapacity.csv")

# Rename columns
df_total.rename(columns={"t": "TECHNOLOGY", "y": "YEAR", "TotalCapacityAnnual": "VALUE"}, inplace=True)
df_resid.rename(columns={"t": "TECHNOLOGY", "y": "YEAR", "ResidualCapacity": "VALUE"}, inplace=True)

# Pivot for plotting
df_total = df_total.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")
df_resid = df_resid.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")
df_new = df_total.sub(df_resid, fill_value=0).clip(lower=0)
years = df_total.index.tolist()

# === Load tech colors ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Create plot ===
fig = go.Figure()
trace_names = []

# Add residual capacity bars (patterned)
for tech in df_resid.columns:
    trace_name = f"{tech} (residual)"
    trace_names.append(trace_name)
    fig.add_trace(go.Bar(
        x=years,
        y=df_resid[tech].fillna(0),
        name=trace_name,
        marker=dict(
            color=tech_colors.get(tech, "#D3D3D3"),
            pattern_shape="/"
        ),
        hovertemplate=f"{tech} (residual)<br>Year: %{{x}}<br>Capacity: %{{y}} GW"
    ))

# Add new capacity bars (solid)
for tech in df_new.columns:
    trace_name = tech
    trace_names.append(trace_name)
    fig.add_trace(go.Bar(
        x=years,
        y=df_new[tech].fillna(0),
        name=trace_name,
        marker=dict(color=tech_colors.get(tech, "#D3D3D3")),
        hovertemplate=f"{tech}<br>Year: %{{x}}<br>New Capacity: %{{y}} GW"
    ))

# === Dropdown visibility filters ===
visible_all = [True] * len(trace_names)
visible_residual_only = [name.endswith("(residual)") for name in trace_names]
visible_new_only = [not name.endswith("(residual)") for name in trace_names]
visible_no_prod = [
    not ("_Prod" in name.replace(" (residual)", "")) for name in trace_names
]

# === Final layout ===
fig.update_layout(
    title="Installed Capacity by Technology (Residual + New)",
    xaxis_title="Year",
    yaxis_title="Capacity (GW)",
    barmode='stack',
    template="plotly_white",
    height=600,
    legend_title="Technology",
    hovermode="x unified",
    updatemenus=[{
        "buttons": [
            {"label": "All", "method": "update", "args": [{"visible": visible_all}]},
            {"label": "Residual Only", "method": "update", "args": [{"visible": visible_residual_only}]},
            {"label": "New Capacity Only", "method": "update", "args": [{"visible": visible_new_only}]},
            {"label": "Exclude _Prod", "method": "update", "args": [{"visible": visible_no_prod}]}
        ],
        "direction": "down",
        "showactive": True,
        "x": 1.05,
        "xanchor": "left",
        "y": 1.15,
        "yanchor": "top"
    }]
)

fig.show()


### Total Annual Production by Technology

In [7]:
import pandas as pd
import plotly.graph_objects as go

# === Load technology color mapping ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Load and rename production data ===
df_prod = pd.read_csv(res_dir / "ProductionByTechnologyAnnual.csv")
df_prod.rename(columns={
    "t": "TECHNOLOGY",
    "y": "YEAR",
    "ProductionByTechnologyAnnual": "VALUE"
}, inplace=True)

# === Pivot for area plot ===
df_pivot = df_prod.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")
years = df_pivot.index.tolist()

# === Create figure ===
fig = go.Figure()
trace_names = []

# Add one stacked area trace per technology
for tech in df_pivot.columns:
    trace_names.append(tech)
    fig.add_trace(go.Scatter(
        x=years,
        y=df_pivot[tech].fillna(0),
        mode='lines',
        stackgroup='one',
        name=tech,
        line=dict(color=tech_colors.get(tech, "#D3D3D3")),  # Use consistent color or fallback
        hovertemplate=f"{tech}<br>Year: %{{x}}<br>Production: %{{y:.2f}} PJ"
    ))

# === Dropdown visibility controls ===
visible_all = [True] * len(trace_names)
visible_none = [False] * len(trace_names)
visible_no_prod = [not tech.endswith("_Prod") for tech in trace_names]

# === Layout and interactivity ===
fig.update_layout(
    title="Annual Electricity Production by Technology",
    xaxis_title="Year",
    yaxis_title="Production (PJ)",
    template="plotly_white",
    legend_title="Technology",
    height=600,
    hovermode="x unified",
    updatemenus=[
        {
            "buttons": [
                {"label": "All", "method": "update", "args": [{"visible": visible_all}]},
                {"label": "None", "method": "update", "args": [{"visible": visible_none}]},
                {"label": "Exclude _Prod", "method": "update", "args": [{"visible": visible_no_prod}]}
            ],
            "direction": "down",
            "showactive": True,
            "x": 1.05,
            "xanchor": "left",
            "y": 1.15,
            "yanchor": "top"
        }
    ]
)

# Show plot
try:
    fig.show()
except ValueError:
    print("🔁 Fallback to browser due to rendering issue.")
    fig.show(renderer="browser")


### Fuel Consumption by Technology

In [8]:
import pandas as pd
import plotly.graph_objects as go

# === Load tech color map ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Load fuel use data ===
df_use = pd.read_csv(res_dir / "UseByTechnology.csv")
df_use.rename(columns={
    "t": "TECHNOLOGY",
    "y": "YEAR",
    "UseByTechnology": "VALUE"
}, inplace=True)

# === Pivot data: fuel consumption by year/technology ===
df_fuel = df_use.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")

# === Create Plotly stacked area chart ===
fig = go.Figure()
for tech in df_fuel.columns:
    fig.add_trace(go.Scatter(
        x=df_fuel.index,
        y=df_fuel[tech].fillna(0),
        mode='lines',
        stackgroup='one',
        name=tech,
        line=dict(color=tech_colors.get(tech, "#D3D3D3")),  # fallback gray
        hovertemplate=f"{tech}<br>Year: %{{x}}<br>Fuel Used: %{{y:.2f}} PJ"
    ))

# === Layout tweaks ===
fig.update_layout(
    title="Annual Fuel Consumption by Technology",
    xaxis_title="Year",
    yaxis_title="Fuel Use (PJ)",
    template="plotly_white",
    legend_title="Technology",
    height=600,
    hovermode="x unified"
)

fig.show()

### Annual Capital Costs by Technology

In [9]:
import pandas as pd
import plotly.graph_objects as go

# === Load technology color map ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Load and prepare capital investment data ===
df_capex = pd.read_csv(res_dir / "CapitalInvestment.csv")
df_capex.rename(columns={"y": "YEAR", "t": "TECHNOLOGY", "CapitalInvestment": "VALUE"}, inplace=True)

# Pivot to wide format
df_pivot = df_capex.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")
years = df_pivot.index.tolist()

# === Build figure ===
fig = go.Figure()

for tech in df_pivot.columns:
    fig.add_trace(go.Bar(
        x=years,
        y=df_pivot[tech].fillna(0),
        name=tech,
        marker=dict(color=tech_colors.get(tech, "#D3D3D3")),  # Fallback to gray
        hovertemplate=f"{tech}<br>Year: %{{x}}<br>Cost: %{{y:.2f}} Million USD"
    ))

# === Layout and interactivity ===
fig.update_layout(
    title="Capital Investment Cost by Technology",
    xaxis_title="Year",
    yaxis_title="Cost (Million USD)",
    barmode='stack',
    template="plotly_white",
    height=600,
    legend_title="Technology",
    hovermode="x unified",
    updatemenus=[
        {
            "buttons": [
                {"label": "All", "method": "update", "args": [{"visible": [True] * len(df_pivot.columns)}]},
                {"label": "None", "method": "update", "args": [{"visible": [False] * len(df_pivot.columns)}]},
            ],
            "direction": "down",
            "x": 1.05,
            "xanchor": "left",
            "y": 1.15,
            "yanchor": "top"
        }
    ]
)

# === Show plot with fallback ===
try:
    fig.show()
except ValueError:
    print("🔁 Fallback to browser due to rendering issue.")
    fig.show(renderer="browser")

### Annual Fixed Costs by Technology

In [10]:
import pandas as pd
import plotly.graph_objects as go

# === Load color mapping ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Load and prepare data ===
df_fixed = pd.read_csv(res_dir / "AnnualFixedOperatingCost.csv")
df_fixed.rename(columns={"y": "YEAR", "t": "TECHNOLOGY", "AnnualFixedOperatingCost": "VALUE"}, inplace=True)

# Pivot for bar plotting
df_pivot = df_fixed.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")
years = df_pivot.index.tolist()

# === Build stacked bar chart ===
fig = go.Figure()

for tech in df_pivot.columns:
    fig.add_trace(go.Bar(
        x=years,
        y=df_pivot[tech].fillna(0),
        name=tech,
        marker=dict(color=tech_colors.get(tech, "#D3D3D3")),  # fallback to light gray
        hovertemplate=f"{tech}<br>Year: %{{x}}<br>Fixed Cost: %{{y:.2f}} Million USD"
    ))

# === Layout ===
fig.update_layout(
    title="Annual Fixed O&M Cost by Technology",
    xaxis_title="Year",
    yaxis_title="Cost (Million USD)",
    barmode='stack',
    template="plotly_white",
    height=600,
    legend_title="Technology",
    hovermode="x unified",
    updatemenus=[
        {
            "buttons": [
                {"label": "All", "method": "update", "args": [{"visible": [True] * len(df_pivot.columns)}]},
                {"label": "None", "method": "update", "args": [{"visible": [False] * len(df_pivot.columns)}]},
            ],
            "direction": "down",
            "x": 1.05,
            "xanchor": "left",
            "y": 1.15,
            "yanchor": "top"
        }
    ]
)

# Show the figure
try:
    fig.show()
except ValueError:
    print("🔁 Fallback to browser due to rendering issue.")
    fig.show(renderer="browser")

### Annual Variable Costs by Technology

In [11]:
import pandas as pd
import plotly.graph_objects as go

# === Load tech color scheme ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Load and prepare variable cost data ===
df_var = pd.read_csv(res_dir / "AnnualVariableOperatingCost.csv")
df_var.rename(columns={
    "y": "YEAR",
    "t": "TECHNOLOGY",
    "AnnualVariableOperatingCost": "VALUE"
}, inplace=True)

# Pivot data for bar plot
df_pivot = df_var.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")
years = df_pivot.index.tolist()

# === Create interactive figure ===
fig = go.Figure()

for tech in df_pivot.columns:
    fig.add_trace(go.Bar(
        x=years,
        y=df_pivot[tech].fillna(0),
        name=tech,
        marker=dict(color=tech_colors.get(tech, "#D3D3D3")),  # fallback to light gray
        hovertemplate=f"{tech}<br>Year: %{{x}}<br>Variable Cost: %{{y:.2f}} Million USD"
    ))

# === Layout settings ===
fig.update_layout(
    title="Annual Variable Operating Cost by Technology",
    xaxis_title="Year",
    yaxis_title="Cost (Million USD)",
    barmode='stack',
    template="plotly_white",
    height=600,
    legend_title="Technology",
    hovermode="x unified",
    updatemenus=[
        {
            "buttons": [
                {"label": "All", "method": "update", "args": [{"visible": [True] * len(df_pivot.columns)}]},
                {"label": "None", "method": "update", "args": [{"visible": [False] * len(df_pivot.columns)}]},
            ],
            "direction": "down",
            "x": 1.05,
            "xanchor": "left",
            "y": 1.15,
            "yanchor": "top"
        }
    ]
)

# === Show figure with fallback ===
try:
    fig.show()
except ValueError:
    print("🔁 Fallback to browser due to rendering issue.")
    fig.show(renderer="browser")

### Seasonality Analysis

In [12]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# === Load tech color map ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Load production by timeslice data ===
df = pd.read_csv(res_dir / "ProductionByTechnology.csv")

# Rename for clarity
df.rename(columns={
    "t": "TECHNOLOGY",
    "l": "TIMESLICE",
    "f": "FUEL",
    "y": "YEAR",
    "ProductionByTechnology": "VALUE"
}, inplace=True)

# Filter only electricity outputs
df = df[df["FUEL"] == "Electricity"]

# === Get list of timeslices and setup subplot grid ===
timeslices = sorted(df["TIMESLICE"].unique())
n_slices = len(timeslices)

fig = make_subplots(
    rows=n_slices, cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{ts} Production" for ts in timeslices],
    vertical_spacing=0.06
)

# === Loop through timeslices and technologies ===
for i, ts in enumerate(timeslices):
    df_slice = df[df["TIMESLICE"] == ts]
    pivot = df_slice.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")

    for tech in pivot.columns:
        fig.add_trace(
            go.Bar(
                x=pivot.index,
                y=pivot[tech].fillna(0),
                name=tech,
                marker=dict(color=tech_colors.get(tech, "#D3D3D3")),
                hovertemplate=f"{tech}<br>Year: %{{x}}<br>Production: %{{y:.2f}} PJ",
                showlegend=(i == 0)  # Show tech once in legend
            ),
            row=i+1, col=1
        )

# === Layout styling ===
fig.update_layout(
    title="Electricity Production by Technology — Subplots by Time Slice",
    height=350 * n_slices,
    barmode="stack",
    template="plotly_white",
    legend_title="Technology",
    hovermode="x unified",
    showlegend=True
)

fig.update_xaxes(title_text="Year", tickangle=-30)
fig.update_yaxes(title_text="Production (PJ)")

fig.show()